In [1]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

import time

In [5]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

URL = "https://www.lge.co.kr/category/tvs"    # 원하는 가전 페이지 링크 걸기

driver.get(URL)

wait = WebDriverWait(driver, 10)

time.sleep(3)

print("페이지 접속 완료:", driver.title)

페이지 접속 완료: TV | LG전자 | 공식몰 LGE.COM


## 더보기 버튼 눌러서 끝까지 펼쳐지는지 확인

In [6]:
click_count = 0

while True:
    try:
        more_button = wait.until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    "//button[contains(@class,'PlpPcButtonMore_more_btn') and .//p[contains(text(),'더보기')]]"
                )
            )
        )

        # 남은 개수 확인
        try:
            remain_text = more_button.find_element(
                By.CSS_SELECTOR,
                "p[class*='PlpPcButtonMore_count']"
            ).text
        except:
            remain_text = "확인불가"

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            more_button
        )

        time.sleep(1)

        driver.execute_script("arguments[0].click();", more_button)

        click_count += 1
        print(f"{click_count}번째 클릭 / 남은 개수: {remain_text}")

        time.sleep(2)

    except:
        print("더보기 버튼 없음 → 끝까지 펼쳐짐")
        print("총 클릭 횟수:", click_count)
        break

1번째 클릭 / 남은 개수: 전체 남은 개수
(161)
2번째 클릭 / 남은 개수: 전체 남은 개수
(131)
3번째 클릭 / 남은 개수: 전체 남은 개수
(101)
4번째 클릭 / 남은 개수: 전체 남은 개수
(71)
5번째 클릭 / 남은 개수: 전체 남은 개수
(41)
6번째 클릭 / 남은 개수: 전체 남은 개수
(11)
더보기 버튼 없음 → 끝까지 펼쳐짐
총 클릭 횟수: 6


## 페이지 펼쳐서 전체 html txt로 저장하는 코드

In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

import time


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 15)

driver.get(URL)

# 🔥 중요: 상품 리스트 로딩 기다리기
wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div[class*='Product']")
    )
)

time.sleep(3)

print("페이지 로딩 완료")


# =========================
# 더보기 클릭
# =========================
click_count = 0

while True:
    try:
        more_button = wait.until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    "//button[contains(@class,'PlpPcButtonMore_more_btn')]"
                )
            )
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            more_button
        )

        time.sleep(1)

        driver.execute_script("arguments[0].click();", more_button)

        click_count += 1
        print(f"{click_count}번째 클릭")

        # 🔥 클릭 후 렌더링 기다림
        time.sleep(3)

    except:
        print("끝까지 펼침 완료")
        break


# 🔥 핵심: 마지막 렌더링 대기
time.sleep(5)


# =========================
# HTML 저장
# =========================
html = driver.page_source

print("HTML 길이:", len(html))  # 🔥 확인 포인트

with open("TV_full_page.html.txt", "w", encoding="utf-8") as f:
    f.write(html)

print("HTML 저장 완료")

driver.quit()

페이지 로딩 완료
1번째 클릭
2번째 클릭
3번째 클릭
4번째 클릭
5번째 클릭
6번째 클릭
끝까지 펼침 완료
HTML 길이: 1237766
HTML 저장 완료


In [ ]:
#확인용
print("HTML 길이:", len(html))
print("상품 카드 개수:", html.count('data-product="plp-product-card"'))
print("상세 링크 예시 포함 여부:", "/tvs/" in html)

HTML 길이: 1237766
상품 카드 개수: 191
상세 링크 예시 포함 여부: True


## 개별 제품 링크 txt 저장

In [11]:
from bs4 import BeautifulSoup

# HTML 읽기
with open("TV_full_page.html.txt", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

links = set()

# 👉 제품 카드 내부 a 태그만 선택
product_cards = soup.find_all("li", attrs={"data-product": "plp-product-card"})

for card in product_cards:
    a_tag = card.find("a", href=True)
    if a_tag:
        href = a_tag["href"]
        full_url = "https://www.lge.co.kr" + href
        links.add(full_url)

# 저장
with open("TV_product_links.txt", "w", encoding="utf-8") as f:
    for link in sorted(links):
        f.write(link + "\n")

print("추출된 링크 개수:", len(links))

추출된 링크 개수: 191


## 한 링크 → 스펙 펼침 → 필요한 값 추출 → CSV 한 행 저장(test)

In [9]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 테스트할 개별 상품 URL 1개
# =========================
url = "https://www.lge.co.kr/tvs/oled48c6kna-stand"

driver.get(url)
wait = WebDriverWait(driver, 10)
time.sleep(3)


# =========================
# 제품 스펙 더 보기 클릭
# =========================
try:
    spec_btn = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//a[contains(@title, '제품스펙 상세정보 펼치기') or contains(@title, '제품 스펙') or contains(., '제품스펙')]"
            )
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        spec_btn
    )
    time.sleep(1)

    driver.execute_script("arguments[0].click();", spec_btn)
    time.sleep(2)

    print("제품 스펙 더 보기 클릭 완료")

except Exception as e:
    print("제품 스펙 더 보기 버튼 없음 또는 이미 펼쳐짐")


# =========================
# HTML 파싱
# =========================
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")


# =========================
# 텍스트 정리 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


# =========================
# 제품명 추출
# =========================
def get_product_name():
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    # h1 안의 span/em 같은 부가정보 제거
    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


# =========================
# 이미지 링크 추출
# =========================
def get_image_url():
    img = soup.select_one("img#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src
    return "https://www.lge.co.kr" + src


# =========================
# 가격 추출
# =========================
def get_price():
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    # blind 텍스트 제거
    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


# =========================
# 스펙 추출 함수
# dt 안에 button이 있어도 get_text로 잡음
# =========================
def get_spec(label):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        if label in dt_text:
            dd = dt.find_next_sibling("dd")
            if dd:
                return clean_text(dd.get_text(" ", strip=True))

    return ""


# =========================
# 한 행 생성
# =========================
row = {
    "제품명": get_product_name(),
    "이미지 링크": get_image_url(),
    "가격(원)": get_price(),
    "에너지 소비효율 등급": get_spec("에너지소비효율등급"),
    "소비전력(W)": get_spec("소비전력"),
    "화면사이즈(cm)": get_spec("화면 사이즈"),
    "해상도": get_spec("해상도"),
    "디스플레이 종류": get_spec("디스플레이 종류"),
    "주사율(Hz)": get_spec("주사율"),
    "운영체제": get_spec("운영체제"),
    "스피커 출력(W)": get_spec("스피커 출력"),
    "스탠드 제외 크기(mm)": get_spec("스탠드 제외크기"),
    "스탠드 제외 무게(kg)": get_spec("스탠드 제외무게"),
}

df = pd.DataFrame([row])
df.to_csv("lge_two_product.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 완료: lge_two_product.csv")
display(df)

제품 스펙 더 보기 클릭 완료
CSV 저장 완료: lge_two_product.csv


,제품명,이미지 링크,가격(원),에너지 소비효율 등급,소비전력(W),화면사이즈(cm),해상도,디스플레이 종류,주사율(Hz),운영체제,스피커 출력(W),스탠드 제외 크기(mm),스탠드 제외 무게(kg)
0,LG 올레드 evo AI (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10770850...,"2,389,000원",3등급,0.5W↓,120 cm,"4K UHD (3,840 x 2,160)",올레드,120Hz (VRR 최대 165Hz),webOS 26,40W,"1,071 x 620 x 46.9",14.9


## 링크 txt 순회 → 스펙 추출 → CSV 누적 저장

In [ ]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
import os

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 링크 txt 읽기
# =========================
with open("TV_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


# =========================
# 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""
    for child in tag.select("span, em"):
        child.decompose()
    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""
    src = img.get("src", "")
    if src.startswith("http"):
        return src
    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""
    for blind in tag.select(".blind"):
        blind.decompose()
    return clean_text(tag.get_text())


def get_spec(soup, label):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))
        if label in dt_text:
            dd = dt.find_next_sibling("dd")
            if dd:
                return clean_text(dd.get_text(" ", strip=True))
    return ""


# =========================
# 3. 전체 순회 + CSV 누적 저장
# =========================
file_path = "lge_tv_all_products.csv"

# 기존 파일 있으면 삭제하고 새로 시작
if os.path.exists(file_path):
    os.remove(file_path)

write_header = True

for idx, url in enumerate(product_links, start=1):
    # print(f"\n[{idx}/{len(product_links)}] 처리중: {url}")

    try:
        driver.get(url)
        time.sleep(3)

        # 스펙 펼치기
        try:
            spec_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//a[contains(@title,'펼치기')]")
                )
            )
            driver.execute_script("arguments[0].click();", spec_btn)
            time.sleep(2)
        except:
            pass

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),

            "에너지 소비효율 등급": get_spec(soup, "에너지소비효율등급"),
            "소비전력(W)": get_spec(soup, "소비전력"),
            "화면사이즈(cm)": get_spec(soup, "화면 사이즈"),
            "해상도": get_spec(soup, "해상도"),
            "디스플레이 종류": get_spec(soup, "디스플레이"),
            "주사율(Hz)": get_spec(soup, "주사율"),
            "운영체제": get_spec(soup, "운영체제"),
            "스피커 출력(W)": get_spec(soup, "스피커 출력"),
            "스탠드 제외 크기(mm)": get_spec(soup, "스탠드 제외크기"),
            "스탠드 제외 무게(kg)": get_spec(soup, "스탠드 제외무게"),
        }

        # 한 행씩 바로 CSV에 누적 저장
        pd.DataFrame([row]).to_csv(
            file_path,
            mode="a",
            header=write_header,
            index=False,
            encoding="utf-8-sig"
        )

        write_header = False

        print("✔ 저장 완료:", row["제품명"])

    except Exception as e:
        print("❌ 실패:", url)
        print(e)


# =========================
# 4. 최종 확인
# =========================
df = pd.read_csv(file_path)

print("\n전체 저장 완료:", file_path)
print("총 수집 개수:", len(df))
display(df.head())

# =========================
# 5. 종료
# =========================
driver.quit()

총 링크 개수: 191

[1/191] 처리중: https://www.lge.co.kr/tvs/100mrgb96bk-stand
✔ 저장 완료: LG Micro RGB evo AI (스탠드형)

[2/191] 처리중: https://www.lge.co.kr/tvs/32lb650bcna-stand
✔ 저장 완료: LG 일반 LED TV (스탠드형)

[3/191] 처리중: https://www.lge.co.kr/tvs/32lb650bena-stand
✔ 저장 완료: LG 일반 LED TV (스탠드형)

[4/191] 처리중: https://www.lge.co.kr/tvs/32lb650bkna-stand
✔ 저장 완료: LG 일반 LED TV (스탠드형)

[5/191] 처리중: https://www.lge.co.kr/tvs/32lq635bcna
✔ 저장 완료: LG 일반 LED TV (스탠드형)

[6/191] 처리중: https://www.lge.co.kr/tvs/32lq635bkna-stand
✔ 저장 완료: LG 일반 LED TV (스탠드형)

[7/191] 처리중: https://www.lge.co.kr/tvs/43lm6350kna-stand
✔ 저장 완료: LG 일반 LED TV (스탠드형)

[8/191] 처리중: https://www.lge.co.kr/tvs/43na1c90aka-stand
✔ 저장 완료: LG 나노셀 AI (스탠드형)

[9/191] 처리중: https://www.lge.co.kr/tvs/43nano80ana-stand
✔ 저장 완료: LG 나노셀 AI (스탠드형)

[10/191] 처리중: https://www.lge.co.kr/tvs/43nano90aka-wall
✔ 저장 완료: LG 나노셀 AI (벽걸이형)

[11/191] 처리중: https://www.lge.co.kr/tvs/43nu850bena-stand


## dataframe으로 누적해서 ->csv로 변환하도록

In [ ]:
%pip install tqdm

In [5]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 링크 txt 읽기
# =========================
with open("TV_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


# =========================
# 3. 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, label):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        if label in dt_text:
            dd = dt.find_next_sibling("dd")
            if dd:
                return clean_text(dd.get_text(" ", strip=True))

    return ""


# =========================
# 4. 전체 링크 순회
# =========================
rows = []
failures = []

pbar = tqdm(product_links, desc="TV 상품 크롤링", unit="개", ncols=100)

for idx, url in enumerate(pbar, start=1):
    try:
        pbar.set_postfix_str(f"{idx}/{len(product_links)} 접속 중")

        driver.get(url)
        time.sleep(3)

        # 제품 스펙 더 보기 클릭
        try:
            spec_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//a[contains(@title,'펼치기')]")
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                spec_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", spec_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),
            "에너지 소비효율 등급": get_spec(soup, "에너지소비효율등급"),
            "소비전력(W)": get_spec(soup, "소비전력"),
            "화면사이즈(cm)": get_spec(soup, "화면 사이즈"),
            "해상도": get_spec(soup, "해상도"),
            "디스플레이 종류": get_spec(soup, "디스플레이"),
            "주사율(Hz)": get_spec(soup, "주사율"),
            "운영체제": get_spec(soup, "운영체제"),
            "스피커 출력(W)": get_spec(soup, "스피커 출력"),
            "스탠드 제외 크기(mm)": get_spec(soup, "스탠드 제외크기"),
            "스탠드 제외 무게(kg)": get_spec(soup, "스탠드 제외무게"),
        }

        rows.append(row)

        pbar.set_postfix_str(f"완료: {row['제품명'][:25]}")

        # 10개마다 임시 저장
        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(
                "lge_tv_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        pbar.set_postfix_str(f"실패: {idx}/{len(product_links)}")


# =========================
# 5. 최종 CSV 저장
# =========================
df = pd.DataFrame(rows)

df.to_csv(
    "lge_tv_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 저장 완료: lge_tv_all_products.csv")
print("총 수집 개수:", len(df))
print("실패 개수:", len(failures))

display(df.head())


# =========================
# 6. 실패 목록 저장
# =========================
if failures:
    pd.DataFrame(failures).to_csv(
        "lge_tv_failures.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("실패 목록 저장 완료: lge_tv_failures.csv")


# =========================
# 7. 드라이버 종료
# =========================
driver.quit()

총 링크 개수: 191


TV 상품 크롤링:   0%|                                                       | 0/191 [00:00<?, ?개/s]

전체 저장 완료: lge_tv_all_products.csv
총 수집 개수: 191
실패 개수: 0


,제품명,이미지 링크,가격(원),에너지 소비효율 등급,소비전력(W),화면사이즈(cm),해상도,디스플레이 종류,주사율(Hz),운영체제,스피커 출력(W),스탠드 제외 크기(mm),스탠드 제외 무게(kg)
0,LG Micro RGB evo AI (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10794841...,"16,500,000원",비대상 (8K / 화면대각선 216cm 초과 모델은 비대상임),0.5W↓,252 cm,"4K UHD (3,840 x 2,160)",Micro RGB,120Hz (VRR 최대 165Hz),webOS 26,40W,"2,230 x 1,277 x 49.9",67.3
1,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10791858...,,1등급,0.5W↓,80 cm,"HD (1,366 x 768)",LED,60Hz,webOS 26,10W,719 x 430 x 71,3.5
2,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10791858...,"450,000원",1등급,0.5W↓,80 cm,"HD (1,366 x 768)",LED,60Hz,webOS 26,10W,719 x 430 x 71,3.5
3,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10791858...,"450,000원",1등급,0.5W↓,80 cm,"HD (1,366 x 768)",LED,60Hz,webOS 26,10W,719 x 430 x 71,3.5
4,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10272836...,"450,000원",1등급,35.4,80 cm,"HD (1,366 x 768)",LED,60Hz,,,736 x 437 x 82.9,4.65


## 사용설명서 추가 버전

In [6]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


with open("TV_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, label):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        if label in dt_text:
            dd = dt.find_next_sibling("dd")
            if dd:
                return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # "제품 사용 설명서(바로보기용)" 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href
                    else:
                        return href



rows = []
failures = []

pbar = tqdm(product_links, desc="TV 상품 크롤링", ncols=120)

for idx, url in enumerate(pbar, start=1):
    try:
        pbar.set_postfix_str(f"{idx}/{len(product_links)} 접속 중")

        driver.get(url)
        time.sleep(3)

        try:
            spec_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//a[contains(@title,'펼치기')]")
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                spec_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", spec_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),
            "에너지 소비효율 등급": get_spec(soup, "에너지소비효율등급"),
            "소비전력(W)": get_spec(soup, "소비전력"),
            "화면사이즈(cm)": get_spec(soup, "화면 사이즈"),
            "해상도": get_spec(soup, "해상도"),
            "디스플레이 종류": get_spec(soup, "디스플레이"),
            "주사율(Hz)": get_spec(soup, "주사율"),
            "운영체제": get_spec(soup, "운영체제"),
            "스피커 출력(W)": get_spec(soup, "스피커 출력"),
            "스탠드 제외 크기(mm)": get_spec(soup, "스탠드 제외크기"),
            "스탠드 제외 무게(kg)": get_spec(soup, "스탠드 제외무게"),
            "제품 사용 설명서": get_manual_pdf_url(soup),
        }

        rows.append(row)

        pbar.set_postfix_str(f"완료: {row['제품명'][:25]}")

        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(
                "lge_tv_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        pbar.set_postfix_str(f"실패: {idx}/{len(product_links)}")


df = pd.DataFrame(rows)

df.to_csv(
    "lge_tv_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 저장 완료: lge_tv_all_products.csv")
print("총 수집 개수:", len(df))
print("실패 개수:", len(failures))

display(df.head())


if failures:
    pd.DataFrame(failures).to_csv(
        "lge_tv_failures.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("실패 목록 저장 완료: lge_tv_failures.csv")


driver.quit()

총 링크 개수: 191


TV 상품 크롤링: 100%|███████████████████████████████████████████| 191/191 [1:27:32<00:00, 27.50s/it, 완료: LG 스마트 캠] 중]/s]

전체 저장 완료: lge_tv_all_products.csv
총 수집 개수: 191
실패 개수: 0


,제품명,이미지 링크,가격(원),에너지 소비효율 등급,소비전력(W),화면사이즈(cm),해상도,디스플레이 종류,주사율(Hz),운영체제,스피커 출력(W),스탠드 제외 크기(mm),스탠드 제외 무게(kg),제품 사용 설명서
0,LG Micro RGB evo AI (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10794841...,"16,500,000원",비대상 (8K / 화면대각선 216cm 초과 모델은 비대상임),0.5W↓,252 cm,"4K UHD (3,840 x 2,160)",Micro RGB,120Hz (VRR 최대 165Hz),webOS 26,40W,"2,230 x 1,277 x 49.9",67.3,https://gscs-b2c.lge.com/open/downloadFile?fil...
1,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10791858...,,1등급,0.5W↓,80 cm,"HD (1,366 x 768)",LED,60Hz,webOS 26,10W,719 x 430 x 71,3.5,https://gscs-b2c.lge.com/open/downloadFile?fil...
2,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10791858...,"450,000원",1등급,0.5W↓,80 cm,"HD (1,366 x 768)",LED,60Hz,webOS 26,10W,719 x 430 x 71,3.5,https://gscs-b2c.lge.com/open/downloadFile?fil...
3,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10791858...,"450,000원",1등급,0.5W↓,80 cm,"HD (1,366 x 768)",LED,60Hz,webOS 26,10W,719 x 430 x 71,3.5,https://gscs-b2c.lge.com/open/downloadFile?fil...
4,LG 일반 LED TV (스탠드형),https://www.lge.co.kr/kr/images/tvs/md10272836...,"450,000원",1등급,35.4,80 cm,"HD (1,366 x 768)",LED,60Hz,,,736 x 437 x 82.9,4.65,https://gscs-b2c.lge.com/open/downloadFile?fil...
